# 🎙️ Học Nhịp Kể Chuyện Ma (Giọng Miễn Phí VieNeu-TTS v3 Turbo + Seed-VC + LoRA)

Pipeline tự động chạy trọn gói trên **Google Colab (khuyên dùng GPU L4 24GB của gói Google AI Ultra — chỉ tốn ~1.5–2 units/giờ)**:

1. **Bước 0:** Trích xuất âm thanh từ file video/audio hoặc link YouTube và tự động cắt đúng **20 phút** kể chuyện (bỏ phần intro).
2. **Bước 1:** Cài **VieNeu-TTS v3 Turbo** (Apache 2.0) → chọn **1 giọng nam trầm miễn phí có sẵn** → xuất file `target_voice.wav` (~10 giây).
3. **Bước 2A:** Dùng **Demucs (`htdemucs`)** tách sạch 100% nhạc nền, chỉ giữ lại giọng nói khô.
4. **Bước 2B:** Dùng **faster-whisper (`large-v3`)** tự động cắt thành 150–250 câu chuẩn **3–15 giây** kèm transcript tiếng Việt đầy đủ dấu câu.
5. **Bước 2C:** Dùng **Seed-VC** chuyển đổi toàn bộ 150–250 câu sang **100% màu giọng miễn phí của VieNeu-TTS**, giữ nguyên nhịp nhanh/chậm, ngắt hơi và nhấn nhá.
6. **Bước 3 & 4:** Train **LoRA cho VieNeu-TTS v3 Turbo**, merge model và đóng gói giọng mới lưu thẳng về **Google Drive**.
7. **Bước 5:** Nghe thử giọng mới và render toàn bộ truyện 20 phút từ file `.txt`!

> ⚠️ **Trước khi bấm chạy:** Vào menu **Runtime → Change runtime type → L4 GPU → Save**. Khi làm xong nhớ vào **Runtime → Disconnect and delete runtime** để không bị trừ Compute Units oan!

In [ ]:
#@title ⚙️ 1. Kết nối Google Drive & Cấu hình Pipeline
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

#@markdown ### 📁 Thư mục lưu toàn bộ dữ liệu & model trên Google Drive
PROJECT_DIR = "/content/drive/MyDrive/horror_voice_lora" #@param {type:"string"}

#@markdown ### 🎬 Nguồn Video/Audio để học nhịp kể (dán link YouTube HOẶC đường dẫn file trên Google Drive)
SOURCE_VIDEO_OR_URL = "https://www.youtube.com/watch?v=SMyvDsc5RfU" #@param {type:"string"}
COOKIES_FILE = "" #@param {type:"string"}
START_TIME = "00:03:00" #@param {type:"string"}
DURATION = "00:20:00" #@param {type:"string"}
FORCE_REEXTRACT = False #@param {type:"boolean"}

#@markdown ### 🎙️ Cấu hình Giọng Mới & Seed-VC
NEW_VOICE_NAME = "Giong_Ke_Chuyen_Ma" #@param {type:"string"}
SEEDVC_F0_CONDITION = True #@param {type:"boolean"}
SEEDVC_DIFFUSION_STEPS = 30 #@param {type:"integer"}
SEEDVC_SEMI_TONE_SHIFT = 0 #@param {type:"integer"}

#@markdown ### 🧠 Cấu hình Train LoRA (VieNeu-TTS v3 Turbo)
LORA_EPOCHS = 3 #@param {type:"integer"}
LORA_RANK = 16 #@param {type:"integer"}

BASE_DIR = Path(PROJECT_DIR)
DIRS = {
    "source": BASE_DIR / "00_source",
    "target": BASE_DIR / "01_target_voice",
    "vocals": BASE_DIR / "02_vocals",
    "clips": BASE_DIR / "03_source_clips",
    "dataset": BASE_DIR / "04_dataset",
    "raw_audio": BASE_DIR / "04_dataset" / "raw_audio",
    "model": BASE_DIR / "05_trained_model",
    "renders": BASE_DIR / "06_renders",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Đã tạo thư mục làm việc trên Google Drive:", BASE_DIR)
!nvidia-smi

## Bước 0: Trích xuất âm thanh từ video & Cắt đoạn kể 20 phút

In [ ]:
#@title ✂️ Trích xuất audio & cắt đúng 20 phút (lưu vào Google Drive)
import shutil
import subprocess
import tempfile
from pathlib import Path
from IPython.display import Audio, display

!pip install -q -U yt-dlp

SOURCE_20M_WAV = DIRS["source"] / "source_20min.wav"


def parse_hms(val: str) -> int:
    val = val.strip()
    if val.isdigit():
        return int(val)
    parts = val.split(":")
    if len(parts) == 2:
        return int(parts[0]) * 60 + int(float(parts[1]))
    if len(parts) == 3:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(float(parts[2]))
    raise ValueError(f"Thời gian không hợp lệ: {val}")


def fmt_hms(sec: int) -> str:
    h, r = divmod(max(0, int(sec)), 3600)
    m, s = divmod(r, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def cut_wav(in_path: Path, out_path: Path, start_s: int, dur_s: int, skip_seek: bool = False):
    cmd = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "warning"]
    if not skip_seek and start_s > 0:
        cmd += ["-ss", fmt_hms(start_s)]
    cmd += ["-i", str(in_path), "-t", str(dur_s), "-vn", "-acodec", "pcm_s16le", "-ar", "44100", "-ac", "2", str(out_path)]
    subprocess.run(cmd, check=True)


if SOURCE_20M_WAV.exists() and not FORCE_REEXTRACT:
    print(f"✅ Đã có sẵn file 20 phút: {SOURCE_20M_WAV}")
else:
    start_s = parse_hms(START_TIME)
    dur_s = parse_hms(DURATION)
    src = SOURCE_VIDEO_OR_URL.strip()

    if src.startswith(("http://", "https://")):
        auth_args = ["--cookies", COOKIES_FILE.strip()] if COOKIES_FILE.strip() else []
        section = f"*{fmt_hms(start_s)}-{fmt_hms(start_s + dur_s)}"
        with tempfile.TemporaryDirectory() as tmpdir:
            tpl = str(Path(tmpdir) / "raw.%(ext)s")
            print(f"Đang tải đoạn {section} bằng yt-dlp...")
            p = subprocess.run([
                "yt-dlp", "-f", "bestaudio/best", "--no-playlist",
                "--download-sections", section, "--force-keyframes-at-cuts",
                "-o", tpl, *auth_args, src,
            ])
            files = [f for f in Path(tmpdir).glob("raw.*") if f.is_file() and f.stat().st_size > 0]
            skip_seek = True
            if p.returncode != 0 or not files:
                print("⚠️ --download-sections không chạy được, chuyển sang tải full audio rồi cắt...")
                p2 = subprocess.run([
                    "yt-dlp", "-f", "bestaudio/best", "--no-playlist",
                    "-o", tpl, *auth_args, src,
                ])
                files = [f for f in Path(tmpdir).glob("raw.*") if f.is_file() and f.stat().st_size > 0]
                if p2.returncode != 0 or not files:
                    raise RuntimeError(
                        "Không tải được video từ YouTube (IP Colab có thể bị yêu cầu xác minh bot). "
                        "Cách xử lý nhanh nhất:\n"
                        "1) Chạy `python3 extract_audio.py` trên máy cá nhân để cắt sẵn `source_20min.wav` rồi upload lên Google Drive và điền đường dẫn vào SOURCE_VIDEO_OR_URL.\n"
                        "2) Hoặc upload file `cookies.txt` lên Google Drive và điền vào COOKIES_FILE."
                    )
                skip_seek = False
            cut_wav(files[0], SOURCE_20M_WAV, start_s, dur_s, skip_seek=skip_seek)
    else:
        local_src = Path(src)
        assert local_src.exists(), f"Không tìm thấy file: {local_src}"
        print(f"Đang trích xuất & cắt {DURATION} từ mốc {START_TIME}...")
        cut_wav(local_src, SOURCE_20M_WAV, start_s, dur_s, skip_seek=False)

# Kiểm tra thời lượng & nghe thử 15 giây đầu
probe = subprocess.check_output([
    "ffprobe", "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", str(SOURCE_20M_WAV)
]).decode().strip()
print(f"🎉 File sẵn sàng: {SOURCE_20M_WAV} | Thời lượng: {float(probe)/60:.2f} phút | Dung lượng: {SOURCE_20M_WAV.stat().st_size/1e6:.1f} MB")

preview_15s = Path("/tmp/preview_source_15s.wav")
subprocess.run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(SOURCE_20M_WAV), "-t", "15", str(preview_15s)], check=True)
print("🔊 Nghe thử 15 giây đầu của đoạn 20 phút vừa cắt:")
display(Audio(str(preview_15s)))

## Bước 1: Cài đặt `VieNeu-TTS v3 Turbo` & Xuất 10 giây "Giọng Đích" miễn phí

In [ ]:
#@title 📦 1A. Cài VieNeu-TTS v3 Turbo (môi trường uv tách biệt) & Liệt kê 25 giọng miễn phí có sẵn
import os
from pathlib import Path

!pip install -q uv
!sudo apt-get -qq update && sudo apt-get -qq install -y espeak-ng > /dev/null

VIENEU_DIR = Path("/content/VieNeu-TTS")
if not VIENEU_DIR.exists():
    !git clone --depth 1 https://github.com/pnnbao97/VieNeu-TTS.git /content/VieNeu-TTS

%cd /content/VieNeu-TTS
!uv sync --extra finetune

list_script = """
import json
from pathlib import Path
from vieneu import Vieneu

tts = Vieneu(mode="v3turbo")
voices = tts.list_preset_voices()
out = []
for idx, v in enumerate(voices):
    if isinstance(v, (list, tuple)) and len(v) >= 2:
        desc, name = str(v[0]), str(v[1])
    elif isinstance(v, dict):
        name = str(v.get("name") or v.get("id") or idx)
        desc = str(v.get("description") or v.get("desc") or name)
    else:
        name = desc = str(v)
    out.append({"index": idx, "name": name, "description": desc})
    print(f"[{idx:02d}] {name}  |  {desc}")

Path("/tmp/vieneu_voices.json").write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
"""
Path("/tmp/list_vieneu_voices.py").write_text(list_script, encoding="utf-8")
!uv run python /tmp/list_vieneu_voices.py

In [ ]:
#@title 🎙️ 1B. Chọn 1 giọng nam trầm miễn phí & Xuất file target_voice.wav (~10 giây)
import json
from pathlib import Path
from IPython.display import Audio, display

#@markdown Để trống `TARGET_PRESET_VOICE` để tự động chọn giọng nam trầm đầu tiên, HOẶC điền đúng tên giọng bạn thấy ở danh sách trên:
TARGET_PRESET_VOICE = "" #@param {type:"string"}

voices_meta = json.loads(Path("/tmp/vieneu_voices.json").read_text(encoding="utf-8"))
chosen_voice = TARGET_PRESET_VOICE.strip()

if not chosen_voice:
    # Ưu tiên tìm giọng nam (male / Nam) trong danh sách dựng sẵn
    male_candidates = [
        v for v in voices_meta
        if any(k in (v["name"] + " " + v["description"]).lower() for k in ("nam", "male", "trầm", "bắc", "miền nam"))
        and "nữ" not in (v["name"] + " " + v["description"]).lower()
        and "female" not in (v["name"] + " " + v["description"]).lower()
    ]
    chosen_voice = (male_candidates[0] if male_candidates else voices_meta[0])["name"]

print(f"✅ Giọng đích miễn phí đã chọn: {chosen_voice}")

TARGET_VOICE_WAV = DIRS["target"] / "target_voice.wav"

gen_target_script = f"""
from vieneu import Vieneu

tts = Vieneu(mode="v3turbo")
sample_text = (
    "Đêm hôm đó, trời đổ cơn mưa rào rất lớn, gió rít từng hồi qua mái tôn cũ kỹ. "
    "Trong căn nhà vắng lặng giữa đồng không mông quạnh, ánh đèn dầu leo lét bỗng nhiên chập chờn rồi tắt phụt."
)
audio = tts.infer(text=sample_text, voice={chosen_voice!r})
tts.save(audio, {str(TARGET_VOICE_WAV)!r})
print("Đã xuất:", {str(TARGET_VOICE_WAV)!r})
"""
Path("/tmp/gen_target_voice.py").write_text(gen_target_script, encoding="utf-8")
%cd /content/VieNeu-TTS
!uv run python /tmp/gen_target_voice.py

print("🔊 Nghe thử 10 giây giọng đích miễn phí (màu giọng mới của kênh bạn):")
display(Audio(str(TARGET_VOICE_WAV)))

## Bước 2A: Tách sạch 100% nhạc nền & tạp âm bằng `Demucs`

In [ ]:
#@title 🧹 Chạy Demucs (htdemucs) tách vocal sạch từ đoạn 20 phút
import shutil
import subprocess
from pathlib import Path
from IPython.display import Audio, display

!pip install -q demucs soundfile

VOCALS_20M_WAV = DIRS["vocals"] / "vocals_20min.wav"

if VOCALS_20M_WAV.exists() and not FORCE_REEXTRACT:
    print(f"✅ Đã có sẵn vocal sạch: {VOCALS_20M_WAV}")
else:
    demucs_out = Path("/content/demucs_out")
    print("Đang tách nhạc nền bằng Demucs trên GPU (mất ~1–2 phút cho 20 phút audio)...")
    subprocess.run([
        "demucs", "-n", "htdemucs", "--two-stems=vocals",
        "-d", "cuda", "-o", str(demucs_out), str(SOURCE_20M_WAV)
    ], check=True)
    raw_vocal = demucs_out / "htdemucs" / SOURCE_20M_WAV.stem / "vocals.wav"
    assert raw_vocal.exists(), f"Không tìm thấy output Demucs tại {raw_vocal}"
    # Chuẩn hoá về 44.1kHz mono sạch cho bước cắt câu & Seed-VC
    subprocess.run([
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "warning",
        "-i", str(raw_vocal), "-acodec", "pcm_s16le", "-ar", "44100", "-ac", "1",
        str(VOCALS_20M_WAV)
    ], check=True)
    shutil.rmtree(demucs_out, ignore_errors=True)
    print(f"🎉 Đã lưu vocal sạch: {VOCALS_20M_WAV}")

preview_v = Path("/tmp/preview_vocal_15s.wav")
subprocess.run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(VOCALS_20M_WAV), "-t", "15", str(preview_v)], check=True)
print("🔊 Nghe thử 15 giây vocal đã tách sạch nhạc nền:")
display(Audio(str(preview_v)))

## Bước 2B: Tự động cắt câu chuẩn `3–15 giây` & Transcribe tiếng Việt bằng `Whisper large-v3`

In [ ]:
#@title 📝 Cắt vocal 20 phút thành 150–250 câu (3.2s – 14.0s) kèm dấu câu chuẩn
import gc
import json
import re
import numpy as np
import soundfile as sf
import torch
from pathlib import Path

!pip install -q faster-whisper

CLIPS_META_PATH = DIRS["clips"] / "clips_meta.json"
MIN_CLIP_SEC = 3.2
MAX_CLIP_SEC = 14.0
PAD_SEC = 0.12

if CLIPS_META_PATH.exists() and not FORCE_REEXTRACT:
    clips_meta = json.loads(CLIPS_META_PATH.read_text(encoding="utf-8"))
    print(f"✅ Đã có sẵn {len(clips_meta)} clip nguồn từ lần chạy trước.")
else:
    from faster_whisper import WhisperModel

    print("Đang tải Whisper large-v3 (float16 trên GPU)...")
    wmodel = WhisperModel("large-v3", device="cuda", compute_type="float16")
    print("Đang nhận diện giọng nói & tách câu tiếng Việt...")
    segments_iter, info = wmodel.transcribe(
        str(VOCALS_20M_WAV),
        language="vi",
        beam_size=5,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=350),
        word_timestamps=True,
        initial_prompt="Xin chào tất cả mọi người, đây là câu chuyện ma rùng rợn... Lúc đó, trời tối đen như mực, bỗng nhiên có tiếng gõ cửa!",
    )

    raw_segs = []
    for seg in segments_iter:
        txt = re.sub(r"\s+", " ", (seg.text or "").strip())
        txt = txt.replace("|", ",")
        if not txt:
            continue
        words = []
        if seg.words:
            for w in seg.words:
                words.append({"start": float(w.start), "end": float(w.end), "word": w.word})
        raw_segs.append({"start": float(seg.start), "end": float(seg.end), "text": txt, "words": words})

    del wmodel
    gc.collect()
    torch.cuda.empty_cache()

    # Tách nhỏ các đoạn đơn lẻ dài hơn MAX_CLIP_SEC theo word timestamps
    normalized_segs = []
    for seg in raw_segs:
        if (seg["end"] - seg["start"]) <= MAX_CLIP_SEC or not seg["words"]:
            normalized_segs.append(seg)
            continue
        cur_words = []
        cur_start = seg["words"][0]["start"]
        for w in seg["words"]:
            if cur_words and (w["end"] - cur_start) > 11.5:
                chunk_txt = re.sub(r"\s+", " ", "".join(x["word"] for x in cur_words).strip())
                if chunk_txt:
                    normalized_segs.append({"start": cur_start, "end": cur_words[-1]["end"], "text": chunk_txt})
                cur_words = [w]
                cur_start = w["start"]
            else:
                cur_words.append(w)
        if cur_words:
            chunk_txt = re.sub(r"\s+", " ", "".join(x["word"] for x in cur_words).strip())
            if chunk_txt:
                normalized_segs.append({"start": cur_start, "end": cur_words[-1]["end"], "text": chunk_txt})

    # Gom các đoạn ngắn thành câu trọn vẹn trong khoảng [MIN_CLIP_SEC, MAX_CLIP_SEC]
    grouped = []
    buf_start = None
    buf_end = None
    buf_texts = []

    def flush_buf():
        nonlocal buf_start, buf_end, buf_texts
        if buf_start is not None and buf_texts:
            dur = buf_end - buf_start
            if MIN_CLIP_SEC <= dur <= MAX_CLIP_SEC:
                text = re.sub(r"\s+", " ", " ".join(buf_texts)).strip()
                if len(text) >= 12:
                    grouped.append({"start": buf_start, "end": buf_end, "text": text})
        buf_start, buf_end, buf_texts = None, None, []

    for seg in normalized_segs:
        if buf_start is None:
            buf_start, buf_end, buf_texts = seg["start"], seg["end"], [seg["text"]]
        else:
            gap = seg["start"] - buf_end
            new_dur = seg["end"] - buf_start
            prev_ends_sentence = buf_texts[-1].endswith((".", "?", "!", "..."))
            if new_dur <= 12.5 and gap <= 1.2 and (not prev_ends_sentence or (buf_end - buf_start) < 6.0):
                buf_end = seg["end"]
                buf_texts.append(seg["text"])
            else:
                flush_buf()
                buf_start, buf_end, buf_texts = seg["start"], seg["end"], [seg["text"]]
    flush_buf()

    # Cắt file WAV nguồn cho từng câu
    audio_data, sr = sf.read(str(VOCALS_20M_WAV), dtype="float32")
    total_len = len(audio_data)
    clips_meta = []

    for idx, item in enumerate(grouped, start=1):
        s_idx = max(0, int((item["start"] - PAD_SEC) * sr))
        e_idx = min(total_len, int((item["end"] + PAD_SEC) * sr))
        clip_wav = audio_data[s_idx:e_idx]
        dur = len(clip_wav) / sr
        if dur < 3.0 or dur > 14.8:
            continue
        fname = f"clip_{idx:04d}.wav"
        sf.write(str(DIRS["clips"] / fname), clip_wav, sr, subtype="PCM_16")
        clips_meta.append({
            "file_name": fname,
            "duration": round(dur, 2),
            "text": item["text"],
        })

    CLIPS_META_PATH.write_text(json.dumps(clips_meta, ensure_ascii=False, indent=2), encoding="utf-8")

total_min = sum(c["duration"] for c in clips_meta) / 60.0
print(f"🎉 Đã cắt xong {len(clips_meta)} clip sạch (3–15s) | Tổng thời lượng: {total_min:.2f} phút")
print("\n--- Xem trước 8 câu đầu tiên ---")
for c in clips_meta[:8]:
    print(f"  {c['file_name']} ({c['duration']}s): {c['text']}")

## Bước 2C: Đổi màu giọng bằng `Seed-VC` sang 100% giọng miễn phí VieNeu (giữ nguyên nhịp kể)

In [ ]:
#@title 🔄 Cài Seed-VC & Chuyển toàn bộ clip sang giọng miễn phí VieNeu-TTS (tự động resume)
import gc
import json
import subprocess
from pathlib import Path
from IPython.display import Audio, display

SEEDVC_DIR = Path("/content/seed-vc")
if not SEEDVC_DIR.exists():
    !git clone --depth 1 https://github.com/Plachtaa/seed-vc.git /content/seed-vc

# Cài các gói phụ thuộc cho Seed-VC mà KHÔNG ghi đè PyTorch CUDA sẵn có của Colab
!pip install -q munch einops pyyaml librosa soundfile pydub scipy tqdm huggingface_hub transformers descript-audio-codec

batch_vc_code = """
import argparse
import json
import os
import sys
import warnings
from pathlib import Path
from tqdm import tqdm
import numpy as np
import librosa
import torch
import torchaudio

warnings.simplefilter("ignore")
os.environ["HF_HUB_CACHE"] = "/content/seed-vc/checkpoints/hf_cache"
sys.path.insert(0, "/content/seed-vc")

from inference import load_models, adjust_f0_semitones, crossfade, device


@torch.no_grad()
def run_batch(args):
    model, semantic_fn, f0_fn, vocoder_fn, campplus_model, mel_fn, mel_fn_args = load_models(args)
    sr = 22050 if not args.f0_condition else 44100
    hop_length = 256 if not args.f0_condition else 512
    max_context_window = sr // hop_length * 30
    overlap_frame_len = 16
    overlap_wave_len = overlap_frame_len * hop_length

    # Chuẩn bị giọng đích (target_voice.wav) MỘT LẦN duy nhất
    ref_audio_np = librosa.load(args.target, sr=sr)[0]
    ref_audio = torch.tensor(ref_audio_np[: sr * 25]).unsqueeze(0).float().to(device)
    ori_waves_16k = torchaudio.functional.resample(ref_audio, sr, 16000)
    S_ori = semantic_fn(ori_waves_16k)
    mel2 = mel_fn(ref_audio)
    target2_lengths = torch.LongTensor([mel2.size(2)]).to(mel2.device)
    feat2 = torchaudio.compliance.kaldi.fbank(
        ori_waves_16k, num_mel_bins=80, dither=0, sample_frequency=16000
    )
    feat2 = feat2 - feat2.mean(dim=0, keepdim=True)
    style2 = campplus_model(feat2.unsqueeze(0))

    if args.f0_condition:
        F0_ori = f0_fn(ori_waves_16k[0], thred=0.03)
        F0_ori = torch.from_numpy(F0_ori).to(device)[None]
        voiced_F0_ori = F0_ori[F0_ori > 1]
        median_log_f0_ori = torch.median(torch.log(voiced_F0_ori + 1e-5))
    else:
        F0_ori = None
        median_log_f0_ori = None

    prompt_condition, _, _, _, _ = model.length_regulator(
        S_ori, ylens=target2_lengths, n_quantizers=3, f0=F0_ori
    )
    max_source_window = max_context_window - mel2.size(2)

    meta = json.loads(Path(args.meta_json).read_text(encoding="utf-8"))
    src_dir = Path(args.source_dir)
    out_dir = Path(args.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    converted_count = 0
    for item in tqdm(meta, desc="Seed-VC converting"):
        fname = item["file_name"]
        out_wav = out_dir / fname
        if out_wav.exists() and out_wav.stat().st_size > 10000:
            converted_count += 1
            continue

        src_wav = src_dir / fname
        if not src_wav.exists():
            continue

        source_np = librosa.load(str(src_wav), sr=sr)[0]
        source_audio = torch.tensor(source_np).unsqueeze(0).float().to(device)
        converted_waves_16k = torchaudio.functional.resample(source_audio, sr, 16000)

        # Clip luôn <= 15s (< 30s) nên 1 forward pass là đủ
        S_alt = semantic_fn(converted_waves_16k)
        mel = mel_fn(source_audio)
        target_lengths = torch.LongTensor([int(mel.size(2) * args.length_adjust)]).to(mel.device)

        if args.f0_condition:
            F0_alt = f0_fn(converted_waves_16k[0], thred=0.03)
            F0_alt = torch.from_numpy(F0_alt).to(device)[None]
            voiced_F0_alt = F0_alt[F0_alt > 1]
            if voiced_F0_alt.numel() > 0:
                log_f0_alt = torch.log(F0_alt + 1e-5)
                median_log_f0_alt = torch.median(torch.log(voiced_F0_alt + 1e-5))
                shifted_log_f0_alt = log_f0_alt.clone()
                if args.auto_f0_adjust:
                    shifted_log_f0_alt[F0_alt > 1] = log_f0_alt[F0_alt > 1] - median_log_f0_alt + median_log_f0_ori
                shifted_f0_alt = torch.exp(shifted_log_f0_alt)
                if args.semi_tone_shift != 0:
                    shifted_f0_alt[F0_alt > 1] = adjust_f0_semitones(shifted_f0_alt[F0_alt > 1], args.semi_tone_shift)
            else:
                shifted_f0_alt = F0_alt
        else:
            shifted_f0_alt = None

        cond, _, _, _, _ = model.length_regulator(
            S_alt, ylens=target_lengths, n_quantizers=3, f0=shifted_f0_alt
        )

        processed_frames = 0
        generated_chunks = []
        previous_chunk = None
        while processed_frames < cond.size(1):
            chunk_cond = cond[:, processed_frames : processed_frames + max_source_window]
            is_last_chunk = processed_frames + max_source_window >= cond.size(1)
            cat_condition = torch.cat([prompt_condition, chunk_cond], dim=1)
            with torch.autocast(device_type=device.type, dtype=torch.float16 if args.fp16 else torch.float32):
                vc_target = model.cfm.inference(
                    cat_condition,
                    torch.LongTensor([cat_condition.size(1)]).to(mel2.device),
                    mel2,
                    style2,
                    None,
                    args.diffusion_steps,
                    inference_cfg_rate=args.inference_cfg_rate,
                )
                vc_target = vc_target[:, :, mel2.size(-1) :]
                vc_wave = vocoder_fn(vc_target.float()).squeeze()
                vc_wave = vc_wave[None, :]

            if processed_frames == 0:
                if is_last_chunk:
                    generated_chunks.append(vc_wave[0].cpu().numpy())
                    break
                generated_chunks.append(vc_wave[0, :-overlap_wave_len].cpu().numpy())
                previous_chunk = vc_wave[0, -overlap_wave_len:]
                processed_frames += vc_target.size(2) - overlap_frame_len
            elif is_last_chunk:
                generated_chunks.append(
                    crossfade(previous_chunk.cpu().numpy(), vc_wave[0].cpu().numpy(), overlap_wave_len)
                )
                break
            else:
                generated_chunks.append(
                    crossfade(previous_chunk.cpu().numpy(), vc_wave[0, :-overlap_wave_len].cpu().numpy(), overlap_wave_len)
                )
                previous_chunk = vc_wave[0, -overlap_wave_len:]
                processed_frames += vc_target.size(2) - overlap_frame_len

        final_wave = torch.tensor(np.concatenate(generated_chunks))[None, :].float()
        torchaudio.save(str(out_wav), final_wave.cpu(), sr)
        converted_count += 1

    print(f"Hoàn tất Seed-VC: {converted_count}/{len(meta)} clips.")


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--meta-json", required=True)
    p.add_argument("--source-dir", required=True)
    p.add_argument("--target", required=True)
    p.add_argument("--output-dir", required=True)
    p.add_argument("--diffusion-steps", type=int, default=30)
    p.add_argument("--length-adjust", type=float, default=1.0)
    p.add_argument("--inference-cfg-rate", type=float, default=0.7)
    p.add_argument("--f0-condition", action="store_true")
    p.add_argument("--auto-f0-adjust", action="store_true")
    p.add_argument("--semi-tone-shift", type=int, default=0)
    p.add_argument("--checkpoint", default=None)
    p.add_argument("--config", default=None)
    p.add_argument("--fp16", action="store_true", default=True)
    run_batch(p.parse_args())
"""
Path("/content/seed-vc/batch_convert.py").write_text(batch_vc_code, encoding="utf-8")

cmd = [
    "python3", "/content/seed-vc/batch_convert.py",
    "--meta-json", str(CLIPS_META_PATH),
    "--source-dir", str(DIRS["clips"]),
    "--target", str(TARGET_VOICE_WAV),
    "--output-dir", str(DIRS["raw_audio"]),
    "--diffusion-steps", str(SEEDVC_DIFFUSION_STEPS),
    "--semi-tone-shift", str(SEEDVC_SEMI_TONE_SHIFT),
]
if SEEDVC_F0_CONDITION:
    cmd += ["--f0-condition", "--auto-f0-adjust"]

%cd /content/seed-vc
subprocess.run(cmd, check=True)

# Tạo file metadata.csv đúng chuẩn VieNeu-TTS (`file_name|text`)
clips_meta = json.loads(CLIPS_META_PATH.read_text(encoding="utf-8"))
valid_lines = []
for item in clips_meta:
    wav_path = DIRS["raw_audio"] / item["file_name"]
    txt = item["text"].replace("|", ",").replace("\n", " ").strip()
    if wav_path.exists() and wav_path.stat().st_size > 10000 and txt:
        valid_lines.append(f"{item['file_name']}|{txt}")

METADATA_CSV = DIRS["dataset"] / "metadata.csv"
METADATA_CSV.write_text("\n".join(valid_lines) + "\n", encoding="utf-8")
print(f"✅ Đã xuất {METADATA_CSV} với {len(valid_lines)} dòng chuẩn VieNeu-TTS.")

# Nghe so sánh Trước vs Sau Seed-VC
print("\n🎧 So sánh 2 câu mẫu (TRƯỚC vs SAU khi đổi sang giọng miễn phí VieNeu):")
for item in clips_meta[:2]:
    print(f"\n📝 {item['file_name']} — {item['text']}")
    print("  [Trước - Vocal nguồn]:")
    display(Audio(str(DIRS["clips"] / item["file_name"])))
    print("  [Sau - Đã đổi sang giọng miễn phí VieNeu]:")
    display(Audio(str(DIRS["raw_audio"] / item["file_name"])))

## Bước 3 & 4: Chuẩn bị Dataset → Train LoRA `VieNeu-TTS v3 Turbo` → Merge & Lưu về Drive

In [ ]:
#@title 🚀 Chuẩn bị train.parquet, Train LoRA, Merge Model & Đóng gói Giọng Mới
import gc
import json
import shutil
import subprocess
import torch
from pathlib import Path

gc.collect()
torch.cuda.empty_cache()

%cd /content/VieNeu-TTS

# 1) Chuẩn bị train.parquet từ bộ dataset trên Google Drive
print("[1/4] Đang mã hoá dataset (prepare_dataset.py)...")
subprocess.run([
    "uv", "run", "python", "finetune/prepare_dataset.py",
    "--dataset-dir", str(DIRS["dataset"]),
], check=True)

TRAIN_PARQUET = DIRS["dataset"] / "train.parquet"
assert TRAIN_PARQUET.exists(), f"Không thấy {TRAIN_PARQUET}"

# 2) Train LoRA + Merge model
RUN_NAME = "horror_voice"
print(f"[2/4] Đang Train LoRA (epochs={LORA_EPOCHS}, r={LORA_RANK}, alpha={LORA_RANK*2})...")
subprocess.run([
    "uv", "run", "python", "finetune/train_lora.py",
    "--data", str(TRAIN_PARQUET),
    "--run", RUN_NAME,
    "--epochs", str(LORA_EPOCHS),
    "--r", str(LORA_RANK),
    "--alpha", str(LORA_RANK * 2),
    "--merge",
], check=True)

MERGED_LOCAL = Path(f"/content/VieNeu-TTS/finetune/output/{RUN_NAME}/merged")
ADAPTER_LOCAL = Path(f"/content/VieNeu-TTS/finetune/output/{RUN_NAME}/adapter")
assert MERGED_LOCAL.exists(), f"Không tìm thấy thư mục merged tại {MERGED_LOCAL}"

# 3) Chọn 1 clip đẹp dài ~6 giây trong dataset đã chuyển giọng để đóng gói speaker embedding
clips_meta = json.loads(CLIPS_META_PATH.read_text(encoding="utf-8"))
best_clip = min(
    [c for c in clips_meta if (DIRS["raw_audio"] / c["file_name"]).exists()],
    key=lambda c: abs(c["duration"] - 6.0),
)
BEST_REF_WAV = DIRS["raw_audio"] / best_clip["file_name"]
print(f"[3/4] Đóng gói giọng `{NEW_VOICE_NAME}` bằng speaker embedding từ {best_clip['file_name']} ({best_clip['duration']}s)...")

subprocess.run([
    "uv", "run", "python", "finetune/make_voice.py",
    "--audio", str(BEST_REF_WAV),
    "--name", NEW_VOICE_NAME,
    "--description", "Nam · Trầm · Kể chuyện ma (LoRA)",
    "--gender", "male",
    "--out", str(MERGED_LOCAL),
], check=True)

# 4) Sao lưu toàn bộ model đã merge & LoRA adapter về Google Drive
MERGED_DRIVE = DIRS["model"] / "merged"
ADAPTER_DRIVE = DIRS["model"] / "adapter"
if MERGED_DRIVE.exists():
    shutil.rmtree(MERGED_DRIVE)
if ADAPTER_DRIVE.exists():
    shutil.rmtree(ADAPTER_DRIVE)

print(f"[4/4] Đang lưu model đã train lên Google Drive: {MERGED_DRIVE} ...")
shutil.copytree(MERGED_LOCAL, MERGED_DRIVE)
if ADAPTER_LOCAL.exists():
    shutil.copytree(ADAPTER_LOCAL, ADAPTER_DRIVE)

print(f"🎉 HOÀN TẤT TRAIN LORA! Model của bạn đã lưu vĩnh viễn tại: {MERGED_DRIVE}")

## Bước 5: Nghe thử giọng mới & Render Full Truyện 20 phút từ file `.txt`

In [ ]:
#@title 🎧 Nghe thử giọng kể chuyện ma mới vừa train xong!
from pathlib import Path
from IPython.display import Audio, display

MERGED_DRIVE = DIRS["model"] / "merged"
TEST_OUT_WAV = DIRS["renders"] / "test_horror_sample.wav"

#@markdown Nhập thử 1 đoạn truyện ma (dùng `...` để ngắt nhịp, có thể chèn `[thở dài]` hoặc `[hắng giọng]`):
TEST_TEXT = "Đêm hôm đó... trời tối đen như mực. [thở dài] Từ phía sau cánh cửa gỗ cũ kỹ, bỗng vang lên ba tiếng gõ chậm rãi... cốc... cốc... cốc. Tôi lạnh sống lưng, không dám thở mạnh, vì rõ ràng trong căn nhà này... chỉ có một mình tôi thôi." #@param {type:"string"}

test_script = f"""
from vieneu import Vieneu

tts = Vieneu(mode="v3turbo", backbone_repo={str(MERGED_DRIVE)!r})
audio = tts.infer(text={TEST_TEXT!r}, voice={NEW_VOICE_NAME!r}, temperature=0.8)
tts.save(audio, {str(TEST_OUT_WAV)!r})
print("Đã xuất file nghe thử:", {str(TEST_OUT_WAV)!r})
"""
Path("/tmp/test_merged_voice.py").write_text(test_script, encoding="utf-8")
%cd /content/VieNeu-TTS
!uv run python /tmp/test_merged_voice.py

display(Audio(str(TEST_OUT_WAV)))

In [ ]:
#@title 📖 (Tuỳ chọn) Render toàn bộ 1 file kịch bản .txt dài 20 phút thành file WAV hoàn chỉnh
from pathlib import Path
from IPython.display import Audio, display

#@markdown Điền đường dẫn file `.txt` trên Google Drive muốn đọc thành truyện:
STORY_TXT_PATH = "" #@param {type:"string"}
OUTPUT_STORY_WAV = str(DIRS["renders"] / "truyen_ma_full.wav") #@param {type:"string"}

if not STORY_TXT_PATH.strip():
    print("💡 Điền đường dẫn file .txt trên Google Drive vào ô STORY_TXT_PATH rồi chạy lại cell này để render cả tập truyện!")
else:
    render_script = rf"""
import re
import numpy as np
import soundfile as sf
from pathlib import Path
from tqdm import tqdm
from vieneu import Vieneu

txt_path = Path({STORY_TXT_PATH.strip()!r})
out_path = Path({OUTPUT_STORY_WAV.strip()!r})
raw_text = txt_path.read_text(encoding="utf-8")

# Tách đoạn & câu hợp lý (tối đa ~220 ký tự / lần gọi để giữ ngữ điệu ổn định nhất)
paragraphs = [p.strip() for p in raw_text.splitlines() if p.strip()]
chunks = []
for p_idx, para in enumerate(paragraphs):
    sentences = re.split(r"(?<=[.!?…])\s+", para)
    buf = ""
    for s in sentences:
        s = s.strip()
        if not s:
            continue
        if len(buf) + len(s) + 1 <= 220:
            buf = (buf + " " + s).strip()
        else:
            if buf:
                chunks.append((buf, 0.45))
            buf = s
    if buf:
        pause = 0.95 if buf.endswith(("...", "…")) else 0.75
        chunks.append((buf, pause))

print(f"Tổng số đoạn cần đọc: {{len(chunks)}}")
tts = Vieneu(mode="v3turbo", backbone_repo={str(DIRS["model"] / "merged")!r})

sr = 48000
pieces = []
for text_chunk, pause_sec in tqdm(chunks, desc="Rendering story"):
    wav = tts.infer(text=text_chunk, voice={NEW_VOICE_NAME!r}, temperature=0.8)
    wav = np.asarray(wav, dtype=np.float32).squeeze()
    pieces.append(wav)
    pieces.append(np.zeros(int(sr * pause_sec), dtype=np.float32))

full_audio = np.concatenate(pieces)
sf.write(str(out_path), full_audio, sr, subtype="PCM_16")
print(f"🎉 Hoàn tất! Đã lưu: {{out_path}} | Thời lượng: {{len(full_audio)/sr/60:.2f}} phút")
"""
    Path("/tmp/render_story.py").write_text(render_script, encoding="utf-8")
    %cd /content/VieNeu-TTS
    !uv run python /tmp/render_story.py